# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huydang2006/flyrank-ML-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question:** Can a transparent rule and a grouped client-holdout model rank visible pages that are likely to under-capture clicks relative to their position tier?

This supports a fixed-capacity human review queue for SEO and content specialists. The unit is one pseudonymized content item at the 2026-04-01 decision point; the output is a ranked list with reason codes and suggested review actions. A wrong positive wastes review capacity, while a wrong negative leaves a potentially reviewable opportunity unseen. The output is decision support, not an automated edit or causal claim.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
OUTPUTS = ROOT / "work" / "outputs"
FIGURES = ROOT / "work" / "figures"
print("Paper source directory:", OUTPUTS)

Paper source directory: d:\EP16\flyrank-ML-internship\work\outputs


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

The analysis uses the FlyRank internship warehouse release and its daily content-performance fact table. The feature window is **2026-01-01 through 2026-03-31**; the future label window is **2026-04-01 through 2026-04-30**. The analytical grain is `(client_hash_id, content_hash_id)` at the 2026-04-01 origin, with one aggregated feature row per content item and client.

Features aggregate search and analytics measurements over the feature window: impressions, clicks, sessions, engaged sessions, position-weighted impressions, position tier, and availability indicators. Rows need at least 100 feature-window impressions and 100 label-window impressions for the evaluated population. Pseudonymous IDs are used only for joins, grouping, and the client split. Trend fields, future-window fields, product/provider flags, client IDs, and content IDs are excluded from score inputs. No client names, addresses, raw queries, or private credentials belong in the paper.

In [2]:
assert (OUTPUTS / "ml07_baseline_metrics.json").exists()
assert (OUTPUTS / "ml08_model_metrics.json").exists()
assert (OUTPUTS / "ml09_validation_metrics.json").exists()
assert (OUTPUTS / "ml10_action_playbook.md").exists()
print("Required public-safe receipts found")

Required public-safe receipts found


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

The baseline flags feature CTR below 0.5% when feature impressions are at least 100 and a valid position tier exists, then ranks by feature impressions. The learned model is logistic regression using eight feature-window measurements: `clicks_90d`, `impressions_90d`, `avg_position_feature`, `sessions_90d`, `engaged_sessions_90d`, `engagement_rate_feature`, `ga4_available_90d`, and `engagement_available`.

The target is 1 when the future-window CTR is below half of the feature-window position-tier expected CTR, subject to the feature and label volume floors and a known position tier. Evaluation uses a grouped-by-client 80/20 split with seed 42: 87,846 training rows across 33 clients and 2,904 held-out rows across 9 clients. ML-09 also reports five-seed grouped validation and time-aware validation.

Leakage checks passed for feature/label time separation, future-field exclusion, identifier exclusion, train/test client disjointness, duplicate grain, and score inputs. Model signal codes describe ranking context; they are not causal explanations.

In [3]:
baseline = json.loads((OUTPUTS / "ml07_baseline_metrics.json").read_text())
model = json.loads((OUTPUTS / "ml08_model_metrics.json").read_text())
validation = json.loads((OUTPUTS / "ml09_validation_metrics.json").read_text())
assert all(baseline["leakage_checks"].values())
assert all(model["leakage_checks"].values())
assert all(validation["leakage_checks"].values())
print("Leakage receipts: PASS")

Leakage receipts: PASS


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

On the same grouped holdout and fixed top-50 review budget, the generated table compares logistic regression, random forest, and the ML-07 rule baseline. Random forest has slightly higher client-balanced Macro Precision@50 than logistic regression, so the learned model is not uniformly superior under every aggregation. The eligible-population base rate is shown from the receipt below.

Across five grouped seeds, logistic-regression P@50 ranged from **0.88 to 0.96**. In the time-aware validation, it ranged from **0.72 to 0.90**. These are measured ranking results, not evidence that an edit causes recovery.

In [4]:
comparison_rows = {row["model"]: row for row in model["comparison"]}
macro_rows = model["client_balanced_comparison"]
method_labels = {
    "logistic_regression": "Logistic regression",
    "random_forest": "Random forest",
    "ml07_rule_baseline": "ML-07 rule baseline",
}
comparison_table = pd.DataFrame([
    {
        "Method": method_labels[name],
        "P@10": row["precision_at_10"],
        "P@20": row["precision_at_20"],
        "P@50": row["precision_at_50"],
        "Macro P@50": macro_rows[name]["macro_precision_at_50"],
    }
    for name, row in comparison_rows.items()
])
assert comparison_rows["logistic_regression"]["precision_at_50"] > comparison_rows["ml07_rule_baseline"]["precision_at_50"]
display(comparison_table.round(2))
print(f"Eligible base rate: {baseline['eligible_base_rate']:.4f}")

,Method,P@10,P@20,P@50,Macro P@50
0,Logistic regression,0.9,0.95,0.94,0.63
1,Random forest,1.0,1.00,0.92,0.64
2,ML-07 rule baseline,0.7,0.75,0.72,0.54


Eligible base rate: 0.5856


## 5. Limitations

*What this work cannot claim.*

- The grouped holdout is client-disjoint but not a randomized experiment; rankings do not establish that edits cause more clicks or CTR.
- Time-aware P@50 varied from 0.72 to 0.90, so performance may shift across periods.
- The eligible population is filtered by future-window impressions, which is an evaluation-population rule and must be disclosed rather than treated as universal coverage.
- The queue is limited by a 50-page review budget and depends on measurable impressions, CTR, and position context.
- Missing engagement data is retained as a signal and a caution, not interpreted as evidence of poor content.
- Pseudonymous warehouse rows do not provide page semantics in this public artifact; human reviewers must inspect the source system before acting.
- The model ranks review candidates. It does not predict a search engine algorithm, infer user intent, or measure business impact.

In [5]:
assert validation["limitations"]
print("Limitations recorded:", len(validation["limitations"]))

Limitations recorded: 4


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

1. Review the first 50 rows per cycle, preserving the primary reason code, supporting signals, confidence note, and model signal note.
2. Confirm impression volume, CTR denominator, position-tier context, title and snippet, intent match, and technical status in the source system.
3. Prefer title/meta and snippet review for high-volume, strong-position, low-CTR rows; use intent-match review where position is deep or model and baseline disagree.
4. Record the evidence, chosen action, and reviewer confidence, then re-measure after a defined observation window.
5. Recheck base rate and Precision@50 each evaluation period. Review thresholds when they move outside the validated range, and retrain only after a new time-aware evaluation preserves feature/label separation.

Do not auto-publish edits, use the queue for personnel evaluation, or act on rows below the denominator floor or without critical context.

In [6]:
playbook = (OUTPUTS / "ml10_action_playbook.md").read_text(encoding="utf-8")
for phrase in ["human review", "No-go list", "Monitoring and retraining triggers"]:
    assert phrase in playbook
print("Action playbook sections found")

Action playbook sections found


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The paper should embed the reproducible validation Precision@50 chart, threshold-sensitivity chart, feature-influence chart, label-distribution chart, and the top-50 action-mix chart. The public queue export contains 2,904 rows and omits pseudonymous client and content identifiers. The results table above is the primary comparison; charts should carry one takeaway sentence and no private identifiers.

In [7]:
required_figures = [
    "validation_precision_at_50.png",
    "threshold_sensitivity.png",
    "feature_importance.png",
    "label_distribution.png",
    "top50_action_mix.png",
]
assert all((FIGURES / name).exists() for name in required_figures)
print("Paper figures found:", len(required_figures))

Paper figures found: 5


## ML-12 closing package

**Five-minute demo outline:** (1) state the review decision and safety boundary; (2) show the feature/label windows and leakage checks; (3) compare the baseline and model at the fixed review budget; (4) show the ranked action queue and human-review checklist; (5) close with validation ranges and limitations.

**Social-post cut:** I built a public-safe ranking aid for SEO review. A grouped client-holdout gave logistic regression Precision@50 of 0.94 versus 0.72 for a transparent CTR-and-volume rule; time-aware validation ranged from 0.72 to 0.90. It prioritizes human review and does not claim that edits cause recovery.

**Employer-facing summary:** Built a reproducible CTR opportunity-scoring workflow on the FlyRank internship warehouse. Compared a transparent baseline with logistic regression using grouped and time-aware validation, with leakage checks and public-safe exports. Delivered a ranked review queue, action playbook, figures, limitations, and rerunnable receipts.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
